# **Rewrite-Retrieve-Read (RRR)**
First rewrites the user's query into a form that is better suited for information retrieval.
It then retrieves relevant information and gives it to the LLM to generate the final answer.

### **Components & Architecture**
- **Query Rewriter:** `ChatOpenAI` (Rewrite Step)
- **Embedding Model:** `OpenAIEmbeddings`
- **Vector Database:** Chroma (Local - Retrieve Step)
- **LLM Model:** `ChatOpenAI` (Read Step)

## **Initial Setup**

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

## **Indexing**

In [ ]:
# load embedding model
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [ ]:
# load data
from langchain.document_loaders import CSVLoader
loader = CSVLoader("./context.csv")
documents = loader.load()

In [ ]:
# split documents
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
documents = text_splitter.split_documents(documents)

In [ ]:
# create vectorstore
from langchain.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents, embeddings)

## **Retriever**

In [ ]:
# create retriever
retriever = vectorstore.as_retriever()

## **RAG Chain**

In [ ]:
# load llm
from langchain_openai import ChatOpenAI
llm = ChatOpenAI()

In [ ]:
# create document chain
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

template = """"
You are a helpful assistant that answers questions based on the following context.
If you don't find the answer in the context, just say that you don't know.
Context: {context}

Question: {input}

Answer:

"""
prompt = ChatPromptTemplate.from_template(template)


rag_chain = (
    {"context": retriever,  "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## **Simple Query**

In [ ]:
# define simple query
simple_query = "who directed the matrix"

In [ ]:
# response
response = rag_chain.invoke(simple_query)
response

## **Distracted Query**

In [ ]:
# define distracted query
distracted_query = "who create the matrics"

In [ ]:
# response
response = rag_chain.invoke(distracted_query)
response

## **Rewrite Retrieve Read**

In [ ]:
# define rewrite prompt for distracted query

template = """Provide a better search query for \
web search engine to answer the given question, end \
the queries with ’**’. Question: \
{x} Answer:"""

rewrite_prompt = ChatPromptTemplate.from_template(template)

In [ ]:
# parse response
def _parse(text):
    return text.strip('"').strip("**")

In [ ]:
# create rewriter chain
rewriter = rewrite_prompt | ChatOpenAI(temperature=0) | StrOutputParser() | _parse

In [ ]:
# updated query
rewriter.invoke({"x": distracted_query})

In [ ]:
# create rewrite retrieve read chain
rewrite_retrieve_read_chain = (
    {
        "context": {"x": RunnablePassthrough()} | rewriter | retriever,
        "input": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# final response
rewrite_retrieve_read_chain.invoke(distracted_query)